# Part 1A — chrY Gene Mutational Burden Analysis
**Sample:** Carl Zimmer Personal Genome  
**Reference:** GRCh37 (hs37d5)  
**Tools:** bcftools · SnpEff · Python  
**Goal:** Identify the top 10 chromosome Y genes by variant count and export their variants to a VCF file.

---
## Pipeline Overview
1. Environment setup & tool verification  
2. Input VCF inspection  
3. Quality control with `bcftools stats`  
4. Functional annotation with `SnpEff`  
5. Parse annotations → gene burden table  
6. Export top-10-gene VCF  
7. Visualisation  


## 1. Environment Setup

- bcftools: `bcftools 1.16`
- java:     openjdk `version "23.0.2-internal" 2025-01-21`
- SnpEff:   /snpEff/snpEff.jar (present)
- Input VCF : zimmer_chrY.vcf
- Output dir: /results

In [ ]:
import subprocess, re, os, sys
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

INPUT_VCF    = "zimmer_chrY.vcf"
SNPEFF_VCF   = "zimmer_chrY_snpeff.vcf"
OUT_DIR      = "/results"
OUT_VCF      = f"{OUT_DIR}/gene_variants_chrY_snpeff.vcf"
OUT_CSV      = f"{OUT_DIR}/top10_genes_chrY_summary.csv"
SNPEFF_DB    = "GRCh37.75"

## 2. Input VCF Inspection

In [2]:
# ── Count header lines and data lines ─────────────────────────────────────
header_lines = int(run(f"grep -c '^#' {INPUT_VCF}"))
data_lines   = int(run(f"grep -cv '^#' {INPUT_VCF}"))
print(f"Header lines : {header_lines}")
print(f"Variant lines: {data_lines:,}")

# ── Show first few header lines (reference build, caller) ─────────────────
header = run(f"grep '^##' {INPUT_VCF} | head -20")
print("\n--- VCF header (first 20 meta-lines) ---")
print(header)

# ── Confirm all variants PASS ──────────────────────────────────────────────
filter_counts = run(f"bcftools view -H {INPUT_VCF} | awk '{{print $7}}' | sort | uniq -c")
print("\n--- FILTER field counts ---")
print(filter_counts)

# ── Chromosome check ───────────────────────────────────────────────────────
chroms = run(f"bcftools view -H {INPUT_VCF} | awk '{{print $1}}' | sort -u")
print("\n--- Chromosomes present ---")
print(chroms)


Header lines : 128
Variant lines: 2,998

--- VCF header (first 20 meta-lines) ---
##fileformat=VCFv4.1
##FILTER=<ID=PASS,Description="All filters passed">
##FILTER=<ID=LowQual,Description="Low quality">
##FILTER=<ID=VQSRTrancheINDEL99.00to99.90,Description="Truth sensitivity tranche level for INDEL model at VQS Lod: -3.3512 <= x < 0.1928">
##FILTER=<ID=VQSRTrancheINDEL99.90to100.00+,Description="Truth sensitivity tranche level for INDEL model at VQS Lod < -870.4591">
##FILTER=<ID=VQSRTrancheINDEL99.90to100.00,Description="Truth sensitivity tranche level for INDEL model at VQS Lod: -870.4591 <= x < -3.3512">
##FILTER=<ID=VQSRTrancheSNP99.00to99.90,Description="Truth sensitivity tranche level for SNP model at VQS Lod: -0.6666 <= x < 1.0923">
##FILTER=<ID=VQSRTrancheSNP99.90to100.00+,Description="Truth sensitivity tranche level for SNP model at VQS Lod < -5429.5747">
##FILTER=<ID=VQSRTrancheSNP99.90to100.00,Description="Truth sensitivity tranche level for SNP model at VQS Lod: -5429.5747 


--- FILTER field counts ---
2998 PASS

--- Chromosomes present ---
Y


## 3. Quality Control — `bcftools stats`

In [3]:
# ── Run bcftools stats ─────────────────────────────────────────────────────
stats_txt = run(f"bcftools stats {INPUT_VCF}")

# ── Parse summary numbers (SN section) ────────────────────────────────────
sn = {}
for line in stats_txt.splitlines():
    if line.startswith("SN"):
        parts = line.split("\t")
        sn[parts[2].rstrip(":")] = parts[3]

print("=== Summary Numbers (SN) ===")
for k, v in sn.items():
    print(f"  {k:<40} {v}")

# ── Parse Ti/Tv ────────────────────────────────────────────────────────────
titv = None
for line in stats_txt.splitlines():
    if line.startswith("TSTV"):
        parts = line.split("\t")
        titv = float(parts[4])
        ts, tv = int(parts[2]), int(parts[3])
        break
print(f"\nTransitions  : {ts:,}")
print(f"Transversions: {tv:,}")
print(f"Ti/Tv ratio  : {titv:.3f}  (expected ~2.1 genome-wide; lower on chrY is normal)")

# ── Parse depth distribution (DP section) ─────────────────────────────────
dp_data = []
for line in stats_txt.splitlines():
    if line.startswith("DP\t"):
        parts = line.split("\t")
        try:
            dp_data.append((int(parts[2]), int(parts[5])))
        except ValueError:
            pass   # skip '>500' bin

dp_df = pd.DataFrame(dp_data, columns=["depth", "count"])
total_dp = dp_df["count"].sum()
cumsum = dp_df["count"].cumsum() / total_dp
p10 = dp_df.loc[(cumsum >= 0.10).idxmax(), "depth"]
p50 = dp_df.loc[(cumsum >= 0.50).idxmax(), "depth"]
p90 = dp_df.loc[(cumsum >= 0.90).idxmax(), "depth"]
mean_dp = (dp_df["depth"] * dp_df["count"]).sum() / total_dp
print(f"\nDepth  mean  : {mean_dp:.1f}×")
print(f"Depth  P10   : {p10}×")
print(f"Depth  P50   : {p50}×")
print(f"Depth  P90   : {p90}×")


=== Summary Numbers (SN) ===
  number of samples                        1
  number of records                        2998
  number of no-ALTs                        0
  number of SNPs                           2998
  number of MNPs                           0
  number of indels                         0
  number of others                         0
  number of multiallelic sites             0
  number of multiallelic SNP sites         0

Transitions  : 1,706
Transversions: 1,292
Ti/Tv ratio  : 1.320  (expected ~2.1 genome-wide; lower on chrY is normal)

Depth  mean  : 26.0×
Depth  P10   : 7×
Depth  P50   : 20×
Depth  P90   : 32×


## 4. Genotype QC — Heterozygous Artifact Analysis

In [4]:
# ── Parse genotypes from VCF ───────────────────────────────────────────────
records = []
with open(INPUT_VCF) as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        fields = line.strip().split("\t")
        pos  = int(fields[1])
        fmt  = fields[8].split(":")
        samp = fields[9].split(":")
        gt   = samp[fmt.index("GT")]
        alleles = gt.replace("|", "/").split("/")
        if "." in alleles:
            gt_class = "missing"
        elif alleles[0] == alleles[1] == "0":
            gt_class = "hom_ref"
        elif alleles[0] == alleles[1]:
            gt_class = "hom_alt"
        else:
            gt_class = "het"
        records.append({"POS": pos, "GT_class": gt_class})

gt_df = pd.DataFrame(records)
counts = gt_df["GT_class"].value_counts()
total  = len(gt_df)

print("=== Genotype class counts ===")
for cls, n in counts.items():
    print(f"  {cls:<12} {n:>6,}  ({100*n/total:.1f}%)")

# ── Heterozygous calls: expected ~0 on hemizygous chrY ────────────────────
het_df = gt_df[gt_df["GT_class"] == "het"].copy()
print(f"\nHet calls: {len(het_df):,} ({100*len(het_df)/total:.1f}%)")
print("These are almost certainly read-mapping artifacts (ampliconic/PAR regions).")

# ── Regional breakdown of het calls ───────────────────────────────────────
bins = [0, 4e6, 6e6, 12e6, 14e6, 22e6, 24e6, 57e6, 60e6, 1e9]
labels = ["0–4 Mb","4–6 Mb (XTR)","6–12 Mb","12–14 Mb (P4/P5)","14–22 Mb",
          "22–24 Mb (P1/P2)","24–57 Mb","57–60 Mb (PAR2-prox)","60+ Mb"]
het_df["region"] = pd.cut(het_df["POS"], bins=bins, labels=labels)
print("\n--- Het calls by region ---")
print(het_df["region"].value_counts().sort_index().to_string())


=== Genotype class counts ===
  hom_alt       2,453  (81.8%)
  het             545  (18.2%)

Het calls: 545 (18.2%)
These are almost certainly read-mapping artifacts (ampliconic/PAR regions).

--- Het calls by region ---
region
0–4 Mb                    6
4–6 Mb (XTR)             19
6–12 Mb                  43
12–14 Mb (P4/P5)        156
14–22 Mb                 13
22–24 Mb (P1/P2)         91
24–57 Mb                 42
57–60 Mb (PAR2-prox)    175
60+ Mb                    0


## 5. Functional Annotation — SnpEff

In [ ]:
snpeff_cmd = (
    f"java -Xmx4g -jar {SNPEFF_JAR} "
    f"-v {SNPEFF_DB} -noStats -canon "
    f"{INPUT_VCF} > {SNPEFF_VCF}"
)
print("Running SnpEff...")
print(f"Command: {snpeff_cmd}\n")

result = subprocess.run(snpeff_cmd, shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print("STDERR:", result.stderr[-3000:])
    raise RuntimeError("SnpEff failed")

n_annotated = int(run(f"grep -cv '^#' {SNPEFF_VCF}"))
print(f"SnpEff annotation complete: {n_annotated:,} variants annotated → {SNPEFF_VCF}")

# ── Show warnings summary ──────────────────────────────────────────────────
warn_lines = [l for l in result.stderr.splitlines() if "WARNING" in l or "ERROR" in l]
print(f"\nWarnings/Errors in SnpEff stderr: {len(warn_lines)}")
for w in warn_lines[:10]:
    print(" ", w)


Running SnpEff...
Command: java -Xmx4g -jar /workspace/snpeff/snpEff/snpEff.jar -v GRCh37.75 -noStats -canon /mnt/user-uploads/zimmer_chrY.vcf > /workspace/zimmer_chrY_snpeff.vcf



SnpEff annotation complete: 2,998 variants annotated → /workspace/zimmer_chrY_snpeff.vcf

Warnings/Errors in SnpEff stderr: 3
  # WARNING!                   : Mitochondrion chromosome 'MT' does not have a mitochondrion codon table (codon table = 'Standard'). You should update the config file.
  WARNINGS: Some warning were detected
  WARNING_TRANSCRIPT_NO_START_CODON	24


## 6. Parse SnpEff ANN= Field

In [ ]:
# ── Consequence types counted as 'strictly genic' ─────────────────────────
# Excludes upstream_gene_variant and downstream_gene_variant (±5 kb flanks)
GENIC_EFFECTS = {
    "intron_variant",
    "intragenic_variant",
    "non_coding_transcript_exon_variant",
    "missense_variant",
    "synonymous_variant",
    "3_prime_UTR_variant",
    "5_prime_UTR_variant",
    "splice_region_variant&non_coding_transcript_exon_variant",
    "splice_region_variant&intron_variant",
    "sequence_feature",
}

# ── Parse ANN= field from SnpEff VCF ──────────────────────────────────────
ann_records = []
with open(SNPEFF_VCF) as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        fields = line.strip().split("\t")
        chrom = fields[0]; pos = int(fields[1])
        ref = fields[3]; alt = fields[4]
        info = fields[7]

        ann_match = re.search(r"ANN=([^;]+)", info)
        if not ann_match:
            continue
        anns = ann_match.group(1).split(",")

        genes_seen = set()
        for ann in anns:
            parts = ann.split("|")
            if len(parts) < 5:
                continue
            effect    = parts[1]
            impact    = parts[2]
            gene_name = parts[3]
            gene_id   = parts[4]
            if not gene_name:
                continue
            # One record per (variant, gene) pair
            if gene_name not in genes_seen:
                genes_seen.add(gene_name)
                ann_records.append({
                    "CHROM": chrom, "POS": pos, "REF": ref, "ALT": alt,
                    "effect": effect, "impact": impact,
                    "gene_name": gene_name, "gene_id": gene_id,
                })

In [ ]:
ann_df = pd.DataFrame(ann_records)
print(f"Total annotation records (variant × gene): {len(ann_df):,}")
print(f"Unique variants with any annotation      : {ann_df['POS'].nunique():,}")

# ── Strictly genic subset ──────────────────────────────────────────────────
genic_strict = ann_df[ann_df["effect"].isin(GENIC_EFFECTS)].copy()
print(f"Strictly genic variants                  : {genic_strict['POS'].nunique():,}")

# ── Intergenic (no gene within ±5 kb) ─────────────────────────────────────
all_pos = set(ann_df["POS"].unique())
genic_pos = set(genic_strict["POS"].unique())
upstream_downstream_pos = set(
    ann_df[ann_df["effect"].isin(["upstream_gene_variant","downstream_gene_variant"])]["POS"]
)
intergenic_pos = all_pos - genic_pos - upstream_downstream_pos
print(f"Upstream/downstream only (±5 kb)         : {len(upstream_downstream_pos - genic_pos):,}")
print(f"Fully intergenic: {len(intergenic_pos):,}")

# ── Consequence type summary ───────────────────────────────────────────────
print("\n=== Consequence type counts (all chrY) ===")
print(ann_df["effect"].value_counts().to_string())

Total annotation records (variant × gene): 3,398
Unique variants with any annotation      : 2,998
Strictly genic variants                  : 491
Upstream/downstream only (±5 kb)         : 241
Fully intergenic                         : 2,266

=== Consequence type counts (all chrY) ===
effect
intergenic_region                                           2500
intron_variant                                               412
downstream_gene_variant                                      200
upstream_gene_variant                                        177
intragenic_variant                                            55
non_coding_transcript_exon_variant                            41
missense_variant                                               3
synonymous_variant                                             3
3_prime_UTR_variant                                            3
splice_region_variant&non_coding_transcript_exon_variant       1
5_prime_UTR_variant                                       

## 7. Gene Burden Ranking

In [ ]:
# ── Count unique variant positions per gene (strictly genic) ──────────────
burden = (
    genic_strict
    .groupby("gene_name")["POS"]
    .nunique()
    .reset_index(name="variant_count")
    .sort_values("variant_count", ascending=False)
    .reset_index(drop=True)
)
burden.index += 1   # 1-based rank
burden.index.name = "rank"

print("=== All genes with ≥5 strictly genic variants ===")
print(burden[burden["variant_count"] >= 5].to_string())

top10 = burden.head(10).copy()

In [ ]:
meta = {
    "PCDH11Y":   {"biotype": "protein_coding", "ensembl_id": "ENSG00000099715",
                  "start": 4868267,  "end": 5610265},
    "NLGN4Y":    {"biotype": "protein_coding", "ensembl_id": "ENSG00000165246",
                  "start": 16634518, "end": 16957530},
    "TTTY14":    {"biotype": "lincRNA",         "ensembl_id": "ENSG00000176728",
                  "start": 21034387, "end": 21239302},
    "UTY":       {"biotype": "protein_coding", "ensembl_id": "ENSG00000183878",
                  "start": 15360259, "end": 15592553},
    "TBL1Y":     {"biotype": "protein_coding", "ensembl_id": "ENSG00000092377",
                  "start": 6778727,  "end": 6959724},
    "PRKY":      {"biotype": "pseudogene",     "ensembl_id": "ENSG00000099725",
                  "start": 7142013,  "end": 7249589},
    "HDHD1P1":   {"biotype": "pseudogene",     "ensembl_id": "ENSG00000234620",
                  "start": 17460542, "end": 17567954},
    "KALP":      {"biotype": "pseudogene",     "ensembl_id": "ENSG00000241859",
                  "start": 15863536, "end": 16027704},
    "TPTE2P4":   {"biotype": "pseudogene",     "ensembl_id": "ENSG00000215506",
                  "start": 28654360, "end": 28725837},
    "LINC00278": {"biotype": "lincRNA",         "ensembl_id": "ENSG00000231535",
                  "start": 2870953,  "end": 2970313},
}

In [ ]:
top10["biotype"]    = top10["gene_name"].map(lambda g: meta[g]["biotype"])
top10["ensembl_id"] = top10["gene_name"].map(lambda g: meta[g]["ensembl_id"])
top10["gene_start"] = top10["gene_name"].map(lambda g: meta[g]["start"])
top10["gene_end"]   = top10["gene_name"].map(lambda g: meta[g]["end"])
top10["length_kb"]  = ((top10["gene_end"] - top10["gene_start"]) / 1000).round(1)

print("\n=== Top 10 chrY Genes by Mutational Burden (SnpEff) ===")
print(top10[["gene_name","ensembl_id","biotype","length_kb","variant_count"]].to_string())

top10.reset_index().to_csv(OUT_CSV, index=False)
print(f"\nSaved: {OUT_CSV}")


=== All genes with ≥5 strictly genic variants ===
       gene_name  variant_count
rank                           
1        PCDH11Y             69
2         NLGN4Y             39
3         TTTY14             24
4            UTY             24
5          TBL1Y             21
6           PRKY             19
7        HDHD1P1             17
8           KALP             17
9        TPTE2P4             15
10     LINC00278             11
11    AC006156.1             11
12         USP9Y             11
13       RFTN1P1             10
14    AC010084.1              9
15           ZFY              8
16      SLC9B1P1              8
17        TTTY11              7
18       TXLNG2P              7
19         STSP1              7
20       ZFY-AS1              7
21          XGPY              6
22    PPP1R12BP1              6
23        BCORP1              6
24         DDX3Y              5
25        TTTY15              5

=== Top 10 chrY Genes by Mutational Burden (SnpEff) ===
      gene_name       ensembl


Saved: /mnt/results/top10_genes_chrY_summary.csv


## 8. Missense Variant Detail

In [ ]:
missense = ann_df[ann_df["effect"] == "missense_variant"][
    ["CHROM","POS","REF","ALT","gene_name","impact"]
].drop_duplicates().sort_values("POS")

print("=== Missense variants (chrY) ===")
print(missense.to_string(index=False))

top10_names = set(top10["gene_name"])
top10_ann   = genic_strict[genic_strict["gene_name"].isin(top10_names)].copy()

effect_label = {
    "intron_variant":                                              "Intronic",
    "intragenic_variant":                                          "Intragenic",
    "non_coding_transcript_exon_variant":                          "ncRNA exon",
    "missense_variant":                                            "Missense",
    "synonymous_variant":                                          "Synonymous",
    "3_prime_UTR_variant":                                         "3' UTR",
    "5_prime_UTR_variant":                                         "5' UTR",
    "splice_region_variant&non_coding_transcript_exon_variant":    "Splice/ncRNA",
    "splice_region_variant&intron_variant":                        "Splice/Intron",
    "sequence_feature":                                            "Other",
}
top10_ann["effect_label"] = top10_ann["effect"].map(effect_label).fillna("Other")

conseq_table = (
    top10_ann.groupby(["gene_name","effect_label"])["POS"]
    .nunique()
    .unstack(fill_value=0)
)
print("\n=== Consequence breakdown per gene (top 10) ===")
print(conseq_table.to_string())


=== Missense variants (chrY) ===
CHROM      POS REF ALT gene_name   impact
    Y  4968368   G   T   PCDH11Y MODERATE
    Y  4968655   T   G   PCDH11Y MODERATE
    Y 14832620   G   T     USP9Y MODERATE

=== Consequence breakdown per gene (top 10) ===
effect_label  Intragenic  Intronic  Missense  Other  Splice/Intron  Synonymous  ncRNA exon
gene_name                                                                                 
HDHD1P1                0        16         0      0              0           0           1
KALP                   0        16         0      0              0           0           1
LINC00278              0        11         0      0              0           0           0
NLGN4Y                 0        38         0      0              0           1           0
PCDH11Y                5        62         2      0              0           0           0
PRKY                   0        18         0      0              0           0           1
TBL1Y                 

## 9. Export Top-10-Gene VCF

In [ ]:
top10_pos_gene = {}   # pos -> gene_name (first hit)
for _, row in genic_strict[genic_strict["gene_name"].isin(top10_names)].iterrows():
    pos = row["POS"]
    if pos not in top10_pos_gene:
        top10_pos_gene[pos] = row["gene_name"]

print(f"Unique variant positions in top 10 genes: {len(top10_pos_gene):,}")

written = 0
with open(SNPEFF_VCF) as fin, open(OUT_VCF, "w") as fout:
    for line in fin:
        if line.startswith("##"):
            fout.write(line)
            # Insert custom INFO header after the last ##INFO line
            if line.startswith("##INFO=<ID=ANN"):
                fout.write('##INFO=<ID=GENE,Number=1,Type=String,Description="Gene name from SnpEff burden analysis">\n')
        elif line.startswith("#CHROM"):
            fout.write(line)
        else:
            fields = line.strip().split("\t")
            pos = int(fields[1])
            if pos in top10_pos_gene:
                gene = top10_pos_gene[pos]
                fields[7] = f"GENE={gene};" + fields[7]
                fout.write("\t".join(fields) + "\n")
                written += 1

print(f"Variants written to {OUT_VCF}: {written:,}")

n_out = int(run(f"grep -cv '^#' {OUT_VCF}"))
assert n_out == written == len(top10_pos_gene), f"Count mismatch: {n_out} vs {written}"
print(f"Integrity check passed: {n_out} variants in output VCF")

gene_tags = run(f"grep -v '^#' {OUT_VCF} | grep -oP 'GENE=\\K[^;]+' | sort | uniq -c | sort -rn")
print("\n--- GENE tag counts in output VCF ---")
print(gene_tags)


Unique variant positions in top 10 genes: 256


Variants written to /mnt/results/gene_variants_chrY_snpeff.vcf: 256
Integrity check passed: 256 variants in output VCF

--- GENE tag counts in output VCF ---
69 PCDH11Y
     39 NLGN4Y
     24 UTY
     24 TTTY14
     21 TBL1Y
     19 PRKY
     17 KALP
     17 HDHD1P1
     15 TPTE2P4
     11 LINC00278


## 10. Visualisation — Figure 1: Top 10 Gene Burden

In [10]:
sns.set_theme(style="ticks", font_scale=1.1)

BIOTYPE_COLORS = {
    "protein_coding": "#0279EE",
    "lincRNA":        "#75A025",
    "pseudogene":     "#FF9400",
}

fig, ax = plt.subplots(figsize=(8, 5))

plot_df = top10.sort_values("variant_count")   # ascending for horizontal bar
colors  = [BIOTYPE_COLORS.get(b, "#888888") for b in plot_df["biotype"]]

bars = ax.barh(plot_df["gene_name"], plot_df["variant_count"],
               color=colors, edgecolor="white", linewidth=0.5)

# Value labels
for bar, val in zip(bars, plot_df["variant_count"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va="center", ha="left", fontsize=10)

ax.set_xlabel("Number of variants (strictly genic)", fontsize=11)
ax.set_title("Top 10 chrY Genes by Mutational Burden\n(SnpEff GRCh37.75, Carl Zimmer genome)", fontsize=12)
ax.set_xlim(0, plot_df["variant_count"].max() * 1.15)
ax.spines[["top","right"]].set_visible(False)

legend_patches = [mpatches.Patch(color=c, label=b) for b, c in BIOTYPE_COLORS.items()]
ax.legend(handles=legend_patches, title="Biotype", loc="lower right", frameon=False)

plt.tight_layout()
fig.savefig(f"{OUT_DIR}/top10_chrY_gene_burden.svg", dpi=150)
fig.savefig(f"{OUT_DIR}/top10_chrY_gene_burden.png", dpi=150)
plt.show()
print("Saved: top10_chrY_gene_burden.svg / .png")


Saved: top10_chrY_gene_burden.svg / .png


## 11. Visualisation — Figure 2: Consequence Breakdown

In [ ]:
CONSEQ_COLORS = {
    "Intronic":     "#56B4E9",   # sky blue
    "Intragenic":   "#E69F00",   # orange
    "ncRNA exon":   "#009E73",   # bluish green
    "Missense":     "#D55E00",   # vermillion
    "Synonymous":   "#CC79A7",   # reddish purple
    "3' UTR":       "#0072B2",   # blue
    "5' UTR":       "#F0E442",   # yellow
    "Splice/Intron":"#000000",   # black
    "Splice/ncRNA": "#999999",   # grey
    "Other":        "#BBBBBB",   # light grey
}

gene_order = top10.sort_values("variant_count", ascending=False)["gene_name"].tolist()
ct = conseq_table.reindex(gene_order)
ct = ct.loc[:, ct.sum() > 0]   # drop empty columns

fig, ax = plt.subplots(figsize=(11, 5))
bottom = np.zeros(len(ct))
for col in ct.columns:
    color = CONSEQ_COLORS.get(col, "#BBBBBB")
    vals = ct[col].values
    bars = ax.bar(range(len(ct)), vals, bottom=bottom, label=col,
                  color=color, edgecolor="white", linewidth=0.5)

    for j, (v, b) in enumerate(zip(vals, bottom)):
        if v >= 3:
            ax.text(j, b + v/2, str(int(v)), ha="center", va="center",
                    fontsize=8, color="white" if col in ("Intronic","Missense","Splice/Intron") else "black",
                    fontweight="bold")
    bottom += vals

ax.set_ylabel("Variant count", fontsize=11)
ax.set_title("Consequence Type Breakdown — Top 10 chrY Genes\n(SnpEff GRCh37.75)", fontsize=12)
ax.set_xticks(range(len(ct)))
ax.set_xticklabels(ct.index, rotation=35, ha="right", fontsize=10)
ax.set_ylim(0, bottom.max() * 1.12)
ax.spines[["top","right"]].set_visible(False)
ax.legend(title="Consequence", bbox_to_anchor=(1.01, 1), loc="upper left",
          frameon=True, fontsize=9, edgecolor="#CCCCCC")

plt.tight_layout()
fig.savefig(f"{OUT_DIR}/consequence_breakdown_chrY.svg", dpi=150, bbox_inches="tight")
fig.savefig(f"{OUT_DIR}/consequence_breakdown_chrY.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: consequence_breakdown_chrY.svg / .png")


Saved: consequence_breakdown_chrY.svg / .png


## 12. Visualisation — Figure 3: Read Depth Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

dp_plot = dp_df[dp_df["depth"] <= 80]
ax.bar(dp_plot["depth"], dp_plot["count"], width=1, color="#0279EE", alpha=0.75, edgecolor="none")

ax.axvline(mean_dp, color="#E74C3C", linestyle="--", linewidth=1.5, label=f"Mean = {mean_dp:.0f}×")
ax.axvline(p50,     color="#FF9400", linestyle="--", linewidth=1.5, label=f"Median = {p50}×")

ax.set_xlabel("Read depth (×)", fontsize=11)
ax.set_ylabel("Number of variants", fontsize=11)
ax.set_title("Read Depth Distribution — chrY Variants", fontsize=12)
ax.spines[["top","right"]].set_visible(False)
ax.legend(frameon=False)

plt.tight_layout()
fig.savefig(f"{OUT_DIR}/depth_distribution_chrY.svg", dpi=150)
fig.savefig(f"{OUT_DIR}/depth_distribution_chrY.png", dpi=150)
plt.show()
print("Saved: depth_distribution_chrY.svg / .png")


Saved: depth_distribution_chrY.svg / .png
